# Final Model Selection and Test-Set Evaluation

## Goal

Select the final forecasting model using validation evidence and evaluate it once on the previously untouched test set.

The project has compared multiple model families and two phases of feature engineering. The strongest validation model is Phase 2 Histogram Gradient Boosting.

The purpose of this notebook is to:

1. Confirm the final model-selection decision.
2. Retrain or restore the selected model using the finalized Phase 2 feature set.
3. Evaluate the selected model on the untouched test period.
4. Report final generalization metrics.
5. Compare validation and test performance.
6. Summarize the final modeling conclusion for the project.

## Final Model Selection

Based on validation performance, the selected model is:

**Phase 2 Histogram Gradient Boosting**

Validation RMSE:

`0.7867`

This model outperformed:

- persistence
- linear regression
- random forest
- Phase 1 Histogram Gradient Boosting
- Phase 1 neural network
- Huber-loss neural network
- reduced-dropout neural network
- Phase 2 neural network

## Phase 2 Neural Network Reference

The strongest Phase 2 neural-network result was:

- MAE: `0.5218`
- RMSE: `0.7973`
- R²: `0.4215`

The neural network remained competitive, but did not outperform Histogram Gradient Boosting.

## Experimental Rule

The test set has remained untouched throughout:

- feature engineering
- model comparison
- neural-network experiments
- hyperparameter decisions
- final model selection

The test set will now be used once to estimate how well the selected model generalizes to unseen future data.

## Plan

1. Load the finalized Phase 2 feature dataset.
2. Recreate the same chronological 70/15/15 split.
3. Recreate the finalized Phase 2 HGB configuration.
4. Train the final model using the established training data.
5. Generate predictions for the untouched test set.
6. Calculate:
   - MAE
   - RMSE
   - R²
7. Compare test performance with validation performance.
8. Inspect actual-vs-predicted behavior on the test period.
9. Analyze whether generalization performance remained stable.
10. Record the final project conclusion and selected model.

## Primary Question

Does the Phase 2 Histogram Gradient Boosting model maintain its validation performance when evaluated on genuinely unseen future data?

## Success Criteria

The final model does not need to reproduce the validation metrics exactly.

A successful result is one where:

- test performance remains reasonably close to validation performance
- there is no evidence of severe overfitting
- the model continues to outperform the original persistence baseline
- the final model choice remains supported by the full experimental evidence

## Final Validation Benchmark

Phase 2 HGB validation performance:

- RMSE: `0.7867`

This value will be compared with the final test-set RMSE to assess generalization.

In [1]:
## 1. Load the Final Phase 2 Dataset

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

data_path = Path(
    "../data/processed/household_power_features_phase2.parquet"
)

df = pd.read_parquet(data_path)
df = df.sort_values("datetime").reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Start:", df["datetime"].min())
print("End:", df["datetime"].max())

Dataset shape: (1950244, 33)
Start: 2006-12-17 17:23:00
End: 2010-11-26 20:02:00


In [2]:
## 2. Recreate the Final Chronological Split

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

print("\nTrain:")
print(train_df["datetime"].min(), "→", train_df["datetime"].max())

print("\nValidation:")
print(val_df["datetime"].min(), "→", val_df["datetime"].max())

print("\nTest:")
print(test_df["datetime"].min(), "→", test_df["datetime"].max())

Train rows: 1365170
Validation rows: 292537
Test rows: 292537

Train:
2006-12-17 17:23:00 → 2009-09-16 02:55:00

Validation:
2009-09-16 02:56:00 → 2010-04-21 22:39:00

Test:
2010-04-21 22:40:00 → 2010-11-26 20:02:00


In [3]:
## define feature set

phase2_feature_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",

    "power_lag_1",
    "power_lag_5",
    "power_lag_15",
    "power_lag_30",
    "power_lag_60",
    "power_lag_120",
    "power_lag_1380",

    "hour",
    "day_of_week",
    "month",
    "is_weekend",

    "rolling_mean_15",
    "rolling_mean_60",
    "rolling_std_60",
    "rolling_mean_1440",

    "power_change_1",
    "power_change_5",
    "power_change_15",
    "power_change_60",

    "rolling_std_15",
    "rolling_max_15",
    "rolling_max_60",
    "rolling_min_60",
    "rolling_range_60"
]

target_col = "power_target_60"

print("Feature count:", len(phase2_feature_cols))

Feature count: 31


In [4]:
## training and test matrices

X_train = train_df[phase2_feature_cols]
y_train = train_df[target_col]

X_test = test_df[phase2_feature_cols]
y_test = test_df[target_col]

print("X train:", X_train.shape)
print("X test:", X_test.shape)

X train: (1365170, 31)
X test: (292537, 31)
